<a href="https://colab.research.google.com/github/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_gpt_to_llama2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Converting a GPT Architecture to Llama 2**

- In this notebook, we will follow Sebastian's approach to converting the original GPT architecture into Llama 2 in a step by step fashion.
- The will primarily be on the main code and unnecessary bloat will be avoided.
- For reference, I used an A100 runtime in this notebook to account for the ~26 GB memory requirement.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

## **1. Setup Notebook**

- Setup the notebook using the `llms-from-scratch` pypi packages.

In [1]:
!pip install llms_from_scratch --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Cannot install llms-from-scratch==0.1.0, llms_from_scratch, ml-dtypes==0.6.0 and tensorflow because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [2]:
from importlib.metadata import version

pkgs = [
    "huggingface_hub",
    "sentencepiece",
    "torch",
]
for p in pkgs:
    print(f"{p} version : {version(p)}")

huggingface_hub version : 1.33.0
sentencepiece version : 0.2.2
torch version : 2.11.0+cu130


In [3]:
import torch
import torch.nn as nn

## **2. Begin Converting GPT Model to Llama**

### 2.1 Replace LayerNorm with RMSNorm Layer

- As shown in the diagram above, we will replace LayerNorm by RMSNorm.
- RMSNorm uses the root mean square to improve computational efficiency, compared to LayerNorm which normalized inputs using mean and variance.
- The RMSNorm operation is as follows, where $x$ is the input $\gamma$ is a trainable parameter (vector), and $\epsilon$ is a small constant to avoid zero-division errors:

$$y_i = \frac{x_i}{\text{RMS}(x)} \gamma_i, \quad \text{where} \quad \text{RMS}(x) = \sqrt{\epsilon + \frac{1}{n} \sum x_i^2}$$



In [4]:
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.emb_dim = emb_dim
        self.weight = nn.Parameter(torch.ones(emb_dim)).float()

    def forward(self, x):
        means = x.pow(2).mean(dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(means + self.eps) # reciprocal sq. root
        return (x_normed * self.weight).to(dtype=x.dtype)
        torch.rsqrt()

In [5]:
torch.manual_seed(123)

eg_batch = torch.randn(2, 3, 4)
rms_norm = RMSNorm(emb_dim=eg_batch.shape[-1])
rms_norm_pytorch = torch.nn.RMSNorm(eg_batch.shape[-1], eps=1e-5)

assert torch.allclose(rms_norm(eg_batch), rms_norm_pytorch(eg_batch))

### **2.2 Replace GeLU with SiLU Activation**

- Llama utilizes the SiLU (also known as the Swish function) activation function instead of GeLU.

$$
\text{silu}(x) = x \cdot \sigma(x), \quad \text{where} \quad \sigma(x) \text{ is the logistic sigmoid.}
$$


In [6]:
class SiLU(nn.Module):
    def __init__(self):
        # Using cleaner syntax
        super().__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

In [7]:
silu = SiLU()
silu_pytorch = torch.nn.SiLU()

assert torch.allclose(silu(eg_batch), silu_pytorch(eg_batch))

### **2.3 Update the FeedForward Module**

- Llama uses a Gates Linear Unit (GLU) variant of SiLU called SwiGLU. The structure of the FeedForward module will need to be changed to accomodated this.
- SwiGLU uses a gating mechanism in the FeedForward layer which has the following formula:

$$\text{SwiGLU}(x) = \text{SiLU}(\text{Linear}_1(x)) * (\text{Linear}_2(x))$$

- Linear Layers 1 and 2 go through an element-wise multiplication while the third layer, i.e. Layer 3,  is applied after the gated activation.


In [8]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # dtype=cfg["dtype"] allows us to load the model directly in lower precision formats.
        # Llama also doesn't use any bias units
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)
        self.silu = SiLU()

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = self.silu(x_fc1) * x_fc2
        return self.fc3(x)


### **2.4 Implement RoPE**

- Compared to traditional "absolute" positional embeddings, Llama uses rotary position embeddings (RoPE), which enables it to capture both absolute and relative positional information simultaneously.
- RoPE can be implemented in two ways:
    - The _split-halves_ version and the _interleaved even/odd version_.
    - Both are mathematically the same as long as we pair dimensions consistently while using the same cos/sin ordering.
- The reference paper for RoPE is [RoFormer: Enhanced Transformer with Rotary Position Embedding (2021)](https://arxiv.org/abs/2104.09864)
- This code uses the RoPE *split-halves* approach, similar to Hugging Face transformers ([modeling_llama.py](https://github.com/huggingface/transformers/blob/e42587f596181396e1c4b63660abf0c736b10dae/src/transformers/models/llama/modeling_llama.py#L173-L188))
- Further information about the The original paper and Meta's official Llama 2 repo, however, use the *interleaved (even/odd)* version ([llama/model.py](https://github.com/meta-llama/llama/blob/6c7fe276574e78057f917549435a2554000a876d/llama/model.py#L64-L74))

In [9]:
def precompute_rope_params(head_dim, theta_base=10_000, context_length=4096):
    assert head_dim % 2 == 0, "head_dim must be divisible by 2 for the RoPE algorithm to work properly."

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0) # Shape: (context_length, head_dim // 2)

    # Expand the angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1) # Shape : (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be divisible by 2 for the RoPE algorithm to work properly."

    # Split x into the first half and the second half
    x1 = x[..., : head_dim // 2] # First half
    x2 = x[..., head_dim // 2 :] # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0) # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

- Sample code for applying RoPE to `q` and `k` tensors.

In [10]:
# Settings
batch_size = 2
context_len = 5
num_heads = 4
head_dim = 16

# Instantiate RoPE params
cos, sin = precompute_rope_params(head_dim=head_dim, context_length=context_len)

# Dummy query and key tensors
torch.manual_seed(123)
queries = torch.randn(batch_size, num_heads, context_len, head_dim)
keys = torch.randn(batch_size, num_heads, context_len, head_dim)

# Apply rotary position embeddings
queries_rot = compute_rope(queries, cos, sin)
keys_rot = compute_rope(keys, cos, sin)


### **2.5 Add RoPE to MultiHeadAttention module**

- Note that GPT applies the positional embeddings to the inputs, whereas Llama applies rotations to the query and the key vectors in the self- attention mechanism.
- We will be modifying the `MultiHeadAttention` class with the appropriate RoPE code. Additionally, the `qkv_bias` option will be removed and `bias=False` will be hardcoded.
- A dtype setting will be added to allow instantiation with a lower precision later.

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dtype=None):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        ################## Key Modifications ###########################
        # set bias = False and dtype=dtype for all linear layers below
        ################################################################
        self.W_query = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, d_out, bias=False, dtype=dtype)
        self.out_proj = nn.Linear(d_out, d_out, bias=False, dtype=dtype) # Linear layer to combine all the head outputs
        # self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

        cos, sin = precompute_rope_params(head_dim=self.head_dim, context_length=context_length)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)


    def forward(self, x):

        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads dimension is added to split the matrix
        # unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        ################## RoPE ######################
        keys = compute_rope(keys, self.cos, self.sin)
        queries = compute_rope(queries, self.cos, self.sin)
        ##############################################

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3) # dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill the attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        # No dropout

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec



In [12]:
### Sample Multihead Attention module's input

# Settings
batch_size = 1
context_len = 100
max_context_len = 4096
embed_dim = 128
num_heads = 4


example_batch = torch.randn((batch_size, context_len, embed_dim))

mha = MultiHeadAttention(
    d_in=embed_dim,
    d_out=embed_dim,
    context_length=max_context_len,
    num_heads=num_heads
)

mha(example_batch)

tensor([[[ 7.7228e-01,  2.4011e-02,  2.4864e-01,  ...,  2.4939e-01,
           8.9658e-02, -1.0373e-01],
         [ 6.1807e-01, -1.9415e-02,  1.3838e-01,  ...,  4.5631e-01,
          -5.0229e-02, -7.2227e-02],
         [ 2.8270e-01, -2.4592e-02, -5.5146e-02,  ...,  3.5184e-01,
          -2.0557e-02, -1.3278e-01],
         ...,
         [-4.4352e-02,  5.5010e-02,  2.8878e-02,  ...,  5.1405e-03,
           3.8362e-02, -1.2888e-03],
         [-4.0955e-02,  5.0045e-02,  2.4860e-02,  ..., -3.5436e-04,
           4.8482e-02,  3.0516e-03],
         [-2.7988e-02,  4.4276e-02,  1.4407e-02,  ..., -1.9056e-03,
           4.1571e-02, -8.0471e-03]]], grad_fn=<UnsafeViewBackward0>)

In [13]:
del mha  # free up memory

### **2.6 Update the TransformerBlock module**

- Replace LayerNorm with RMSNorm, remove `dropout` and `qkv_bias` and add the `dtype` setting.

In [14]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dtype=cfg["dtype"],
        )
        self.ff = FeedForward(cfg)

        ############# Modification ##############
        self.norm1 = RMSNorm(cfg["emb_dim"])
        self.norm2 = RMSNorm(cfg["emb_dim"])
        ##########################################

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x) # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut # Add the original input back

        return x

### **2.7 Update the model class**

- Time to update the model code surrounding the `TransformerBlock`.
- We will replace absolute positional embeddings for `RoPE`, `LayerNorm` with `RMSNorm`. `dropout` will be removed while `dtype` setting will be added.

In [15]:
class Llama2Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        # abs. pos. emb. and dropout removed
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # Layer norm removed and RMSNorm added
        self.final_norm = RMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

## **3. Initialize The Model**

In [16]:
# We will load the config file for the 1.5B parameter GPT model
GPT_CONFIG_1558M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1600,
    "n_heads": 25,
    "n_layers": 48,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [17]:
LLAMA2_CONFIG_7B = {
    "vocab_size": 32000,
    "context_length": 4096,
    "emb_dim": 4096,
    "n_heads": 32,
    "n_layers": 32,
    "hidden_dim": 11008, # Size of the intermediate dimension in FeedForward
    "dtype": torch.bfloat16
}

In [18]:
# Initialize the model
model = Llama2Model(LLAMA2_CONFIG_7B)

In [19]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

Total number of parameters: 6,738,415,616


In [20]:
# Function to calculate the memory requirements of the model
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0

    for param in model.parameters():
        # Calculate the total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if grads are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate the buffer size
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (number of elements) * (size of each element in bytes)
    # Assume parameters and grads are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024 ** 3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

float32 (PyTorch default): 52.33 GB
bfloat16: 26.17 GB


In [21]:
# Transfer model to GPU if available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

model.to(device);

## **4. Load Tokenizer**

In [ ]:
#del mha  # free up memorymport os
import os
from huggingface_hub import login

try:
    # Try to retrieve the token securely from Google Colab Secrets
    from google.colab import userdata
    access_token = userdata.get('HF_Llama')
except Exception:
    # Fallback to checking environment variables or asking manually
    access_token = os.getenv('HF_Llama')

if access_token:
    login(token=access_token)
else:
    # If no token is found, trigger the interactive login prompt
    login()

In [25]:
from huggingface_hub import hf_hub_download

tokenizer_file = hf_hub_download(
    repo_id="meta-llama/Llama-2-7b",
    filename="tokenizer.model",
    local_dir="Llama-2-7b"
)

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

In [26]:
import sentencepiece as spm


class LlamaTokenizer:
    def __init__(self, tokenizer_file):
        sp = spm.SentencePieceProcessor()
        sp.load(tokenizer_file)
        self.tokenizer = sp

    def encode(self, text):
        return self.tokenizer.encode(text, out_type=int)

    def decode(self, ids):
        return self.tokenizer.decode(ids)


tokenizer = LlamaTokenizer(tokenizer_file)

In [28]:
from previous_chapters import generate, token_ids_to_text, text_to_token_ids

torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort movesαllRadius deletingpretcc否']; future eer napulate lackус während inter DES издаSchéon로жа Bass differencespadxsnu ;; ctx始


- As expected, the output doesn't make any sense as Llama 2 hasn't been trained yet.
- Next we will load pretrained weights for the model.

## **5. Load Pretrained Weights**

- Loading the base model's weights.

In [30]:
weights_file = hf_hub_download(
   repo_id="meta-llama/Llama-2-7b",
   filename="consolidated.00.pth",
   local_dir="Llama-2-7b"
)

consolidated.00.pth: reconstructing file:   0%|          |  0.00B / 13.5GB            

consolidated.00.pth: downloading bytes:           |  0.00B            

In [31]:
weights = torch.load(weights_file, weights_only=True)

In [32]:
# Show the first 15 tensors from the model weights
list(weights.keys())[:15]

['tok_embeddings.weight',
 'norm.weight',
 'output.weight',
 'layers.0.attention.wq.weight',
 'layers.0.attention.wk.weight',
 'layers.0.attention.wv.weight',
 'layers.0.attention.wo.weight',
 'layers.0.feed_forward.w1.weight',
 'layers.0.feed_forward.w2.weight',
 'layers.0.feed_forward.w3.weight',
 'layers.0.attention_norm.weight',
 'layers.0.ffn_norm.weight',
 'layers.1.attention.wq.weight',
 'layers.1.attention.wk.weight',
 'layers.1.attention.wv.weight']

In [36]:
def assign(left, right, tensor_name="unknown"):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch for {tensor_name}. Left: {left.shape}, Right: {right.shape}")

    with torch.no_grad():
        if isinstance(right, torch.Tensor):
            left.copy_(right)
        else:
            left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

    return left

def permute(w: torch.Tensor, n_heads, out_dim, in_dim):
    return (w.view(n_heads, out_dim // n_heads // 2, 2, in_dim)
             .transpose(1, 2)       # put axis 2 next to heads)
             .reshape(out_dim, in_dim))

In [37]:
def load_weights_into_llama(model, param_config, params):

    cfg = LLAMA2_CONFIG_7B

    model.tok_emb.weight = assign(model.tok_emb.weight, params["tok_embeddings.weight"])

    for l in range(param_config["n_layers"]):

        # The original Meta/Llama checkpoints store Q and K so that the two numbers
        # that form one complex RoPE pair sit next to each other inside the head dimension ("sliced" layout).
        # Our RoPE implementation, similar to the one in Hugging Face, expects an interleaved layout
        # For example, with n_heads=2 and head_dim = 8
        #                         ┌── pair 0 ──┐      ┌── pair 1 ──┐
        # Meta (sliced):    [ h0:  r0 r1 r2 r3,   h1:  r0 r1 r2 r3  ]
        # Ours & HF (interleaved):  [ h0: r0 r0 r1 r1 r2 r2 r3 r3  , h1: ... ]
        # For more information, please see the discussion in the PR: https://github.com/rasbt/LLMs-from-scratch/pull/747

        # So, below, for q_raw and k_raw, we must re‑order the checkpoint weights using the slices_to_interleave helper

        q_raw = params[f"layers.{l}.attention.wq.weight"]
        model.trf_blocks[l].att.W_query.weight = assign(
            model.trf_blocks[l].att.W_query.weight,
            permute(q_raw, cfg["n_heads"], cfg["emb_dim"], cfg["emb_dim"])
        )
        k_raw = params[f"layers.{l}.attention.wk.weight"]
        model.trf_blocks[l].att.W_key.weight = assign(
            model.trf_blocks[l].att.W_key.weight,
            permute(k_raw, cfg["n_heads"], cfg["emb_dim"], cfg["emb_dim"])
        )
        model.trf_blocks[l].att.W_value.weight = assign(
            model.trf_blocks[l].att.W_value.weight,
            params[f"layers.{l}.attention.wv.weight"]
        )
        model.trf_blocks[l].att.out_proj.weight = assign(
            model.trf_blocks[l].att.out_proj.weight,
            params[f"layers.{l}.attention.wo.weight"]
        )
        model.trf_blocks[l].norm1.weight = assign(
            model.trf_blocks[l].norm1.weight,
            params[f"layers.{l}.attention_norm.weight"]
        )

        # Load FeedForward weights
        model.trf_blocks[l].ff.fc1.weight = assign(
            model.trf_blocks[l].ff.fc1.weight,
            params[f"layers.{l}.feed_forward.w1.weight"]
        )
        # For some reason w2 and w3 are provided in the wrong order in the weights file
        model.trf_blocks[l].ff.fc2.weight = assign(
            model.trf_blocks[l].ff.fc2.weight,
            params[f"layers.{l}.feed_forward.w3.weight"]
        )
        model.trf_blocks[l].ff.fc3.weight = assign(
            model.trf_blocks[l].ff.fc3.weight,
            params[f"layers.{l}.feed_forward.w2.weight"]
        )
        model.trf_blocks[l].norm2.weight = assign(
            model.trf_blocks[l].norm2.weight,
            params[f"layers.{l}.ffn_norm.weight"]
        )

    # Load output layer weights
    model.final_norm.weight = assign(model.final_norm.weight, params["norm.weight"])
    model.out_head.weight = assign(model.out_head.weight, params["output.weight"])


load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

In [39]:
# Generating text
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("In the grim darkness", tokenizer).to(device),
    max_new_tokens=100,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 In the grim darkness of the far future, there is only war.
The Warhammer 40,000: Dawn of War series is a real-time strategy game set in the Warhammer 40,000 universe. The series is based on the tabletop game of the same name.
The first game in the series, Dawn of War, was released in 2004. The game was well-received by critics and players alike.


## **6. Instruction Fine-Tuned Model**

In [40]:
del model

In [42]:
weights_file = hf_hub_download(
   repo_id="meta-llama/Llama-2-7b-chat",
   filename="consolidated.00.pth",
   local_dir="Llama-2-7b-chat"
)

weights = torch.load(weights_file, weights_only=True)
model = Llama2Model(LLAMA2_CONFIG_7B)
load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

consolidated.00.pth: reconstructing file:   0%|          |  0.00B / 13.5GB            

consolidated.00.pth: downloading bytes:           |  0.00B            

In [44]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids("What happens in the grim darkness of Warhammer?", tokenizer).to(device),
    max_new_tokens=150,
    context_size=LLAMA2_CONFIG_7B["context_length"],
    top_k=1,
    temperature=0.
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 What happens in the grim darkness of Warhammer?

In the grim darkness of the far future, humanity has spread across the galaxy, colonizing countless worlds and moons. But with their vast empire comes the threat of war, corruption, and the eternal struggle for survival.

On the planet of Ultramar, the Imperium of Man has established a stronghold, a bulwark against the encroaching darkness. The Adeptus Mechanicus, a powerful tech-priesthood, maintains the Imperium's war machines and technology, while the Adeptus Astartes, the Space Marines, defend the Imperium against its enemies.

But even as they fight to maintain order
